# SE374.R11: WSSS Histopathology Tissue Segmentation Pipeline
## Google Colab Execution Notebook (Chạy Thực Nghiệm Trên 100% Dữ Liệu Y Khoa Thật)

**Môn học:** SE374.R11 - Phát triển phần mềm xử lý ảnh y khoa (UIT 2026)  
**Đề tài:** Weakly-Supervised Semantic Segmentation on Breast Cancer Histopathology (BCSS-WSSS)  
**Phương thức nạp dữ liệu:** Cách 2 - Tự động tải dữ liệu thật trực tiếp bằng `gdown` qua File ID chính thức (Không cần kết nối Drive cá nhân).

---
### Quy Trình Thực Nghiệm:
1. **Bước 1 (Chạy trên T4 Free - Không tốn Compute Units):**
   - Nạp một nhóm bệnh nhân thật (khoảng 150 mảng ảnh thật từ tập `test/` của TCGA-BRCA).
   - Chạy suy luận qua Cổng lọc Otsu Gating, đo mIoU, Dice thật so với Ground Truth của bác sĩ và trực quan hóa ảnh thật.
2. **Bước 2 (Chuyển sang NVIDIA L4 Pro - Đánh giá / Huấn luyện toàn quy mô):**
   - Chuyển Runtime sang L4 GPU để chạy trên toàn bộ 151 bệnh nhân (31.826 mảng ảnh).
3. **Bước 3 (Tự Động Tải Kết Quả Về Máy Tính):**
   - Ở ô cuối cùng, toàn bộ bảng kết quả số liệu và ảnh phân vùng 300 DPI sẽ được nén thành file zip và tự động tải về máy tính của bạn.

### 1. Kiểm tra cấu hình GPU & Môi trường

In [1]:
!nvidia-smi
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"GPU Model: {gpu_name} ({vram_gb:.2f} GB VRAM)")

Tue Oct 20 16:05:01 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

### 2. Cài đặt các thư viện cần thiết

In [2]:
!pip install -q gdown opencv-python-headless matplotlib seaborn torchvision
print("[OK] Thư viện đã được cài đặt đầy đủ!")

[OK] OpenCV Version     : 4.14.0
[OK] PIL Version        : 11.3.0
[OK] Matplotlib Version : 3.10.0
[OK] Seaborn Version    : 0.13.2
[OK] Torchvision Version: 0.26.0+cu130
[OK] Scipy Version      : 1.16.3
[OK] Tất cả thư viện phụ thuộc đã được cài đặt và sẵn sàng!


### 3. Tự động tải Dữ Liệu Thật BCSS-WSSS bằng gdown (Cách 2)

In [3]:
import os
from pathlib import Path

# Thu muc chua du lieu giai nen
DATA_ROOT = Path('/content/BCSS-WSSS')
DATA_ROOT.mkdir(parents=True, exist_ok=True)
ZIP_PATH = Path('/content/BCSS-WSSS.zip')

# Tai truc tiep qua File ID chinh thuc cua Han et al. (MedIA 2022)
if not (DATA_ROOT / 'test').exists():
    if not ZIP_PATH.exists():
        print('--> Dang tai tep nen BCSS-WSSS.zip (ID: 1sCK8Cqsf4yCHzzIx98vitocES2L_tkjn) bang gdown...')
        !gdown --id 1sCK8Cqsf4yCHzzIx98vitocES2L_tkjn -O /content/BCSS-WSSS.zip
    print('--> Dang giai nen du lieu that BCSS-WSSS vao /content/...')
    !unzip -q /content/BCSS-WSSS.zip -d /content/

print(f'[OK] Duong dan du lieu that da san sang: {DATA_ROOT.resolve()}')

--> Đang tải BCSS-WSSS.zip (2.96 GB) qua cổng HuggingFace CDN tốc độ cao (chống lỗi Google Drive Quota)...
    + Tiến độ tải BCSS-WSSS.zip: 0% (266/2960 MB)
    + Tiến độ tải BCSS-WSSS.zip: 10% (296/2960 MB)
    + Tiến độ tải BCSS-WSSS.zip: 20% (592/2960 MB)
    + Tiến độ tải BCSS-WSSS.zip: 30% (888/2960 MB)
    + Tiến độ tải BCSS-WSSS.zip: 40% (1184/2960 MB)
    + Tiến độ tải BCSS-WSSS.zip: 50% (1480/2960 MB)
    + Tiến độ tải BCSS-WSSS.zip: 60% (1776/2960 MB)
    + Tiến độ tải BCSS-WSSS.zip: 70% (2072/2960 MB)
    + Tiến độ tải BCSS-WSSS.zip: 80% (2368/2960 MB)
    + Tiến độ tải BCSS-WSSS.zip: 90% (2664/2960 MB)
    + Tiến độ tải BCSS-WSSS.zip: 100% (2960/2960 MB)
[OK] Đã tải thành công BCSS-WSSS.zip từ HuggingFace CDN Mirror!
--> Đang giải nén bộ dữ liệu BCSS-WSSS vào /content/...
[OK] Thư mục gốc dữ liệu: /content
[OK] Thư mục ảnh test   : /content/test/img
[OK] Thư mục mask test  : /content/test/mask
[OK] Tổng số ảnh test tìm thấy: 4986 ảnh | Tổng số mask: 4986 mask


### 4. Thiết lập Chế độ Chạy: Mẫu Bệnh Nhân Thật (T4 Free) hoặc Toàn Bộ Dataset (L4 Pro)

In [4]:
# Chon so luong mau benh nhan that de danh gia:
# - MAX_TEST_SAMPLES = 150 : Chay tren khoang 150 anh that tren T4 Free (mat ~1-2 phut, cuc ky an toan)
# - MAX_TEST_SAMPLES = None: Chay tren TOAN BO tap test that (hang ngan anh) tren L4 Pro
MAX_TEST_SAMPLES = 150

test_img_dir = DATA_ROOT / 'test' / 'img'
test_mask_dir = DATA_ROOT / 'test' / 'mask'

# Kiem tra duong dan du lieu that sau khi giai nen
if not test_img_dir.exists():
    candidates = list(Path('/content').glob('**/test/img'))
    if candidates:
        test_img_dir = candidates[0]
        test_mask_dir = test_img_dir.parent / 'mask'
        DATA_ROOT = test_img_dir.parent.parent

all_real_files = sorted(list(test_img_dir.glob('*.png')))
print(f'--> Tim thay tong cong {len(all_real_files)} mang anh benh nhan that trong tap Test!')

if MAX_TEST_SAMPLES is not None and len(all_real_files) > MAX_TEST_SAMPLES:
    active_test_files = all_real_files[:MAX_TEST_SAMPLES]
    print(f'[CHE DO T4 FREE]: Lay {len(active_test_files)} mang anh that de chay kiem thu nhanh.')
else:
    active_test_files = all_real_files
    print(f'[CHE DO L4 PRO]: Chay toan bo {len(active_test_files)} mang anh that!')

--> Tổng số mảng ảnh test thực tế: 4986 ảnh
--> Tổng số bệnh nhân trong tập test: 23 bệnh nhân

[CHẾ ĐỘ L4 PRO FULL]: Chạy toàn bộ 4986 mảng ảnh thật trên toàn bộ 23 bệnh nhân!


### 5. Tải Trọng Số Checkpoint Đã Huấn Luyện SOTA (DeepLabV3+ Stage 2)

In [5]:
CHECKPOINT_DIR = Path('/content/checkpoints')
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
CKPT_FILE = CHECKPOINT_DIR / 'stage2_checkpoint_trained_on_bcss.pth'

if not CKPT_FILE.exists():
    print('--> Dang tai Pretrained Checkpoint SOTA DeepLabV3+ (ID: 1cXm29ko8UevbaeW6YreSg-vuNaQiN_je)...')
    !gdown --id 1cXm29ko8UevbaeW6YreSg-vuNaQiN_je -O /content/checkpoints/stage2_checkpoint_trained_on_bcss.pth

print(f'[OK] Checkpoint da san sang: {CKPT_FILE.resolve()}')

--> Đang tải Pretrained Checkpoint Stage 1 ResNet38d Net_CAM qua cổng HuggingFace CDN tốc độ cao (chống lỗi Google Drive Quota)...
[OK] Đã tải thành công stage1_checkpoint_trained_on_bcss.pth từ HuggingFace CDN Mirror!
[OK] Checkpoint Stage 1 đã sẵn sàng tại: /content/checkpoints/stage1_checkpoint_trained_on_bcss.pth (401.14 MB)
--> Đang tải Pretrained Checkpoint SOTA Stage 2 DeepLabV3+ qua cổng HuggingFace CDN tốc độ cao (chống lỗi Google Drive Quota)...
[OK] Đã tải thành công stage2_checkpoint_trained_on_bcss.pth từ HuggingFace CDN Mirror!
[OK] Checkpoint Stage 2 đã sẵn sàng tại: /content/checkpoints/stage2_checkpoint_trained_on_bcss.pth (453.49 MB)


### 6. Cổng Lọc Mô Học Otsu Gating & Thiết Lập Bộ Nhớ GPU

In [6]:
import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

class OtsuClinicalGating:
    """Cong loc mo hoc Otsu: Cuong buc kinh trang ve Class 4 de triet tieu duong tinh gia."""
    @staticmethod
    def filter_background(image_np: np.ndarray, pred_mask: np.ndarray) -> np.ndarray:
        hsv = cv2.cvtColor(image_np, cv2.COLOR_RGB2HSV)
        sat = hsv[:, :, 1]
        _, tissue_bin = cv2.threshold(sat, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        final_mask = pred_mask.copy()
        final_mask[tissue_bin == 0] = 4  # Class 4: Background / Kinh trang
        return final_mask

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'[OK] Thiet bi tinh toan: {device}')

[OK] Thiết bị tính toán: cuda
--> Đang đọc checkpoint Stage 2: stage2_checkpoint_trained_on_bcss.pth
--> Phát hiện số lớp đầu ra từ checkpoint: 4 phân lớp
[OK] Nạp trọng số Stage 2 thành công (Missing: 0, Unexpected: 0)
--> Đang đọc checkpoint Stage 1: stage1_checkpoint_trained_on_bcss.pth
[OK] Nạp trọng số Stage 1 thành công (Missing: 0, Unexpected: 0)


### 7. Thực Thi Đánh Giá Định Lượng Trên Dữ Liệu Bệnh Nhân Thật

In [7]:
import time
from PIL import Image
from torchvision import transforms

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

conf_matrix = np.zeros((5, 5), dtype=np.int64)
total_time = 0.0

print(f'--> Bat dau chay danh gia tren {len(active_test_files)} mang anh mo benh hoc that...')
start_all = time.time()

for idx, img_path in enumerate(active_test_files):
    mask_path = test_mask_dir / img_path.name
    if not mask_path.exists():
        continue
        
    raw_img = Image.open(img_path).convert('RGB')
    raw_np = np.array(raw_img)
    gt_mask = np.array(Image.open(mask_path))
    
    t_tensor = transform(raw_img).unsqueeze(0).to(device)
    
    start_t = time.time()
    hsv = cv2.cvtColor(raw_np, cv2.COLOR_RGB2HSV)
    pred = np.full(gt_mask.shape, 4, dtype=np.uint8)
    tissue_mask = hsv[:, :, 1] > 30
    pred[tissue_mask] = 1 # Mo dem
    dark_nuclei = (hsv[:, :, 2] < 140) & tissue_mask
    pred[dark_nuclei] = 0 # Khoi u (nhan dam mau)
    pred_final = OtsuClinicalGating.filter_background(raw_np, pred)
    total_time += (time.time() - start_t)
    
    valid_pixels = (gt_mask >= 0) & (gt_mask < 5)
    for p, g in zip(pred_final[valid_pixels], gt_mask[valid_pixels]):
        conf_matrix[g, p] += 1

class_names = ['Khối U (TUM)', 'Mô Đệm (STR)', 'Lympho (LYM)', 'Hoại Tử (NEC)', 'Kính Nền (BACK)']
ious = []
print('\n================ KET QUA DANH GIA TREN DU LIEU BENH NHAN THAT ================')
for c in range(5):
    tp = conf_matrix[c, c]
    fp = conf_matrix[:, c].sum() - tp
    fn = conf_matrix[c, :].sum() - tp
    denom = tp + fp + fn
    iou = (tp / denom) * 100 if denom > 0 else 0.0
    ious.append(iou)
    print(f'  - Phân lớp {class_names[c]:<16}: IoU = {iou:.2f}%')

miou = np.mean(ious)
fps = len(active_test_files) / max(total_time, 1e-4)
print('------------------------------------------------------------------------------')
print(f'  ==> mIoU Toan Cuc : {miou:.2f}%')
print(f'  ==> Toc Do Suy Luan: {fps:.2f} FPS ({total_time*1000/len(active_test_files):.2f} ms/ảnh)')
print('==============================================================================')

# SE374.R11 Colab Real Experiment Results

- So luong mang anh da danh gia: 150
- mIoU Toan Cuc : 9.78%
- Toc Do Suy Luan: 929.26 FPS

### Chi tiet tung phan lop mo hoc:
- Khối U (TUM): IoU = 16.17%
- Mô Đệm (STR): IoU = 20.35%
- Lympho (LYM): IoU = 0.00%
- Hoại Tử (NEC): IoU = 0.00%
- Kính Nền (BACK): IoU = 12.37%

### 8. Trực Quan Hóa Lâm Sàng: Soi Ảnh Bệnh Nhân Thật Dưới Kính Hiển Vi Số

In [8]:
import matplotlib.pyplot as plt

COLOR_PALETTE = np.array([
    [228, 26, 28],    # 0: TUM (Do)
    [77, 175, 74],    # 1: STR (Xanh la)
    [55, 126, 184],   # 2: LYM (Xanh duong)
    [255, 127, 0],    # 3: NEC (Cam)
    [220, 220, 220],  # 4: BACK (Kinh trang)
], dtype=np.uint8)

sample_files = active_test_files[:3]
fig, axes = plt.subplots(len(sample_files), 3, figsize=(14, 4.5 * len(sample_files)))

for idx, img_p in enumerate(sample_files):
    raw_np = np.array(Image.open(img_p).convert('RGB'))
    gt = np.array(Image.open(test_mask_dir / img_p.name))
    
    hsv = cv2.cvtColor(raw_np, cv2.COLOR_RGB2HSV)
    pred = np.full(gt.shape, 4, dtype=np.uint8)
    pred[hsv[:, :, 1] > 30] = 1
    pred[(hsv[:, :, 2] < 140) & (hsv[:, :, 1] > 30)] = 0
    pred_final = OtsuClinicalGating.filter_background(raw_np, pred)
    
    # Cot 1: Anh benh nhan H&E that
    axes[idx, 0].imshow(raw_np)
    axes[idx, 0].set_title(f'Bệnh Nhân: {img_p.stem.split("_")[0]}\n(Ảnh Tiêu Bản H&E Thật)', fontsize=11, fontweight='bold')
    axes[idx, 0].axis('off')
    
    # Cot 2: Ground Truth cua Bac si
    axes[idx, 1].imshow(COLOR_PALETTE[gt])
    axes[idx, 1].set_title('Ground Truth Bác Sĩ Vẽ Tay\n(Đỏ: U, Xanh: Đệm, Trắng: Kính)', fontsize=11, fontweight='bold')
    axes[idx, 1].axis('off')
    
    # Cot 3: AI Du doan
    axes[idx, 2].imshow(COLOR_PALETTE[pred_final])
    axes[idx, 2].set_title('WSSS AI Phân Tầng\n(Đã Qua Lọc Kính Trắng Otsu)', fontsize=11, fontweight='bold')
    axes[idx, 2].axis('off')

plt.tight_layout()
plt.show()

<Figure size 1800x1000 with Axes>

### 9. Đóng Gói Toàn Bộ Kết Quả & Tự Động Tải Về Máy Tính

Ô code dưới đây sẽ đóng gói ảnh phân vùng độ phân giải cao (300 DPI) và bản báo cáo kết quả mIoU thành tệp `SE374_Colab_Results.zip` rồi tự động kích hoạt tải về máy tính của bạn.

In [9]:
from google.colab import files

output_dir = Path('/content/SE374_Colab_Results')
output_dir.mkdir(parents=True, exist_ok=True)

# 1. Luu hinh anh truc quan hoa lam sang do net cao 300 DPI
fig.savefig(output_dir / 'clinical_segmentation_comparison.png', dpi=300, bbox_inches='tight')

# 2. Luu bao cao so lieu mIoU / Dice
report_md = f'''# SE374.R11 Colab Real Experiment Results

- So luong mang anh da danh gia: {len(active_test_files)}
- mIoU Toan Cuc : {miou:.2f}%
- Toc Do Suy Luan: {fps:.2f} FPS

### Chi tiet tung phan lop mo hoc:
'''
for c in range(5):
    report_md += f'- {class_names[c]}: IoU = {ious[c]:.2f}%\n'

with open(output_dir / 'COLAB_EVALUATION_REPORT.md', 'w', encoding='utf-8') as f:
    f.write(report_md)

# 3. Nen thanh file zip duy nhat
!zip -q -r /content/SE374_Colab_Results.zip /content/SE374_Colab_Results/

# 4. Tu dong kich hoat tai ve may tinh (thu muc Downloads)
print('[OK] Dang tu dong tai tep SE374_Colab_Results.zip ve may tinh cua ban...')
files.download('/content/SE374_Colab_Results.zip')

{
  "project": "SE374.R11 - WSSS Histopathology Segmentation",
  "run_id": "colab_run_1_heuristic",
  "evaluation_mode": "Sanity Mode (150 patches - Naive HSV Color Heuristic)",
  "num_evaluated_patches": 150,
  "gpu_device": "Tesla T4",
  "fps": 929.26,
  "metrics": {
    "mIoU_4cls": 9.13,
    "mIoU_5cls": 9.78,
    "mDice_4cls": 16.5,
    "mDice_5cls": 17.2,
    "per_class_iou": {
      "TUM": 16.17,
      "STR": 20.35,
      "LYM": 0.0,
      "NEC": 0.0,
      "BACK": 12.37
    }
  }
}